# Kaggle 2xT4 status + append-submit debug

entity_id=kaggle_2xt4_status_submit_append_debug; type=notebook; purpose=verify_solver_status_owner_and_incremental_submission_append

In [ ]:
import os, sys, json, time, pathlib, shutil, subprocess, threading, queue
from datetime import datetime

def now():
    return datetime.now().strftime('%H:%M:%S')

def gpu_snapshot():
    try:
        out = subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'], text=True, stderr=subprocess.STDOUT).strip()
        return out.replace('\n', ' | ')
    except Exception as exc:
        return f'nvidia-smi-unavailable:{type(exc).__name__}:{exc}'

def run_live(cmd, *, env=None, cwd=None, title='', heartbeat_sec=30, timeout_sec=None):
    if title:
        print('\n' + '=' * 96, flush=True)
        print(f'[{now()}] {title}', flush=True)
        print('=' * 96, flush=True)
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    merged.setdefault('PYTHONUNBUFFERED', '1')
    print(f'[{now()}] cmd={cmd}', flush=True)
    print(f'[{now()}] cwd={cwd or os.getcwd()}', flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd or os.getcwd()), env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    q = queue.Queue()
    def reader():
        for line in iter(proc.stdout.readline, ''):
            q.put(line.rstrip('\n'))
    threading.Thread(target=reader, daemon=True).start()
    lines, start, last_hb, last_out = [], time.time(), 0.0, time.time()
    while True:
        while True:
            try:
                line = q.get_nowait()
            except queue.Empty:
                break
            lines.append(line)
            last_out = time.time()
            print(line, flush=True)
        rc = proc.poll()
        elapsed = time.time() - start
        if timeout_sec is not None and elapsed > timeout_sec and rc is None:
            proc.kill()
            raise TimeoutError(f'timeout after {elapsed:.1f}s')
        if rc is not None:
            while True:
                try:
                    line = q.get_nowait()
                except queue.Empty:
                    break
                lines.append(line)
                print(line, flush=True)
            print(f'[{now()}] process_exit | returncode={rc} | elapsed_sec={elapsed:.1f}', flush=True)
            text = '\n'.join(lines)
            if rc != 0:
                raise subprocess.CalledProcessError(rc, cmd, output=text)
            return text
        if time.time() - last_hb >= heartbeat_sec:
            print(f'[{now()}] still_running | elapsed_sec={elapsed:.1f} | silent_for_sec={time.time()-last_out:.1f} | gpu=[{gpu_snapshot()}]', flush=True)
            last_hb = time.time()
        time.sleep(0.5)

print('python', sys.version)
print('gpu', gpu_snapshot())

In [ ]:
PROJECT_DIR = pathlib.Path('/kaggle/working/CayleyBeam100H100')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
input_root = pathlib.Path('/kaggle/input')
candidates = []
if input_root.exists():
    candidates = [p for p in input_root.rglob('*') if p.is_dir() and (p / 'beam_engine.py').exists()]
assert candidates, 'project dataset with beam_engine.py not found'
shutil.copytree(candidates[0], PROJECT_DIR)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('DATASET_DIR_SELECTED', candidates[0])
print('PROJECT_DIR', PROJECT_DIR)
print('files_ok', (PROJECT_DIR/'beam_engine.cpp').exists(), (PROJECT_DIR/'beam_kernels.cu').exists(), (PROJECT_DIR/'scripts'/'solve_testcsv_2gpu.py').exists())

In [ ]:
env = {
    'PYTHONUNBUFFERED': '1',
    'CUDA_VISIBLE_DEVICES': '0,1',
    'USE_CUDA_GRAPHS': '1',
    'INFERENCE_BACKEND': 'fullbeamnice_static',
    'INFERENCE_PARALLELISM': '1',
    'GLOBAL_BEAM_WIDTH': '65536',
    'MAX_DEPTH': '4',
    'KNOWN_SCRAMBLE': 'U,R',
    'B_MICRO': '8192',
    'K_EXPAND_TILE': '8192',
    'SCORE_RING_DEPTH': '8',
    'NET_RING_DEPTH': '2',
    'BUCKET_CAP_PER_PEER': '65536',
    'BETA': '1.20',
    'HASH_LOAD_FACTOR': '0.45',
    'PROBE_LIMIT': '256',
    'HISTOGRAM_PERIOD_MICRO': '2',
    'LOG_EVERY': '1',
    'DEPTH_LOG_EVERY': '1',
    'SUBMISSION_APPEND_EACH': '1',
    'SUBMISSION_PATH': '/kaggle/working/submission.csv',
    'NCCL_IB_DISABLE': '1',
    'NCCL_P2P_DISABLE': '1',
    'NCCL_SOCKET_IFNAME': 'lo',
    'GLOO_SOCKET_IFNAME': 'lo',
    'NCCL_DEBUG': 'WARN',
    'BUILD_VERBOSE': '0',
}
print(json.dumps(env, indent=2, ensure_ascii=False))
run_live([sys.executable, '-u', 'scripts/t4_sizing.py'], env=env, cwd=PROJECT_DIR, title='sizing', heartbeat_sec=10, timeout_sec=120)
out = run_live([sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone', '--nnodes=1', '--nproc_per_node=2', 'scripts/solve_testcsv_2gpu.py'], env=env, cwd=PROJECT_DIR, title='2xT4 known scramble status+append test', heartbeat_sec=15, timeout_sec=900)
assert 'SAMPLE_RESULT' in out, 'SAMPLE_RESULT missing'
assert 'SUBMISSION_WRITTEN' in out, 'SUBMISSION_WRITTEN missing'
submission = pathlib.Path('/kaggle/working/submission.csv')
text = submission.read_text(encoding='utf-8')
print('SUBMISSION_CONTENT')
print(text)
assert text.startswith('initial_state_id,path\n'), 'submission header missing'
assert len([x for x in text.splitlines() if x.strip()]) >= 2, 'submission row missing'
print('STATUS_APPEND_TEST_OK', json.dumps({'submission': str(submission), 'lines': len(text.splitlines())}, ensure_ascii=False))